# Multi-turn RL on MILES using Qwen3-1.7B on CodeContests

This notebook runs entirely in one environment: the Megatron/SGLang trainer, the Harbor agent server, and JupyterLab are all started here, and every service talks to the others over `localhost`.

Harbor normally grades each task in its own container. Since this environment has no nested-container runtime, it instead uses a modified Harbor that grades each task directly on the host, sandboxed with `proot`.

Because training and grading both take a while, the notebook starts them as background processes rather than running them in a cell that blocks until they finish. You'll watch their progress through log tails and a monitor cell instead. Paths throughout the notebook come from the environment (`REPO_DIR`, `EX`, `WORK_DIR`, `HARBOR_TASKS_DIR`), and the CodeContests dataset is already included, so there's nothing to download before you begin.

## 0. Prerequisites

Before you start, you should have this JupyterLab session running inside the workshop environment, with a single idle GPU visible to the container. You don't need to prepare the dataset yourself: the CodeContests tasks and their difficulty splits are already baked in, at `$HARBOR_TASKS_DIR` and `$EX/data/cc_train_*.jsonl`. That means §4 below, which normally handles data preparation, will simply do nothing unless you explicitly force it to re-run.

In [ ]:
import os, subprocess

# Paths come from the environment (preset when the workshop was launched).
REPO_DIR  = os.environ.get("REPO_DIR") or subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True, cwd=os.getcwd()).strip()
EX        = os.environ.get("EX",  os.path.join(REPO_DIR, "examples/experimental/qwen3-codecontests"))
RUNTIME   = os.path.join(EX, "runtime")
WORK_DIR  = os.environ.get("WORK_DIR",  os.path.join(RUNTIME, "work"))
TASKS_DIR = os.environ.get("HARBOR_TASKS_DIR", "/workspace/harbor_tasks_cc")
HF_CACHE  = os.environ.get("HF_HOME", os.path.join(RUNTIME, "cache/hf"))
WANDB_KEY = os.environ.get("WANDB_KEY", "")

os.environ.update(REPO_DIR=REPO_DIR, EX=EX, RUNTIME=RUNTIME, WORK_DIR=WORK_DIR,
                  HARBOR_TASKS_DIR=TASKS_DIR, HF_HOME=HF_CACHE, WANDB_KEY=WANDB_KEY)
for _d in (WORK_DIR, HF_CACHE):
    os.makedirs(_d, exist_ok=True)

for _k in ("REPO_DIR", "EX", "WORK_DIR", "HARBOR_TASKS_DIR", "HF_HOME"):
    print(f"{_k:16}= {os.environ[_k]}")
print(f"WANDB_KEY       = {'set' if WANDB_KEY else 'unset (W&B disabled)'}")
print(f"baked tasks     = {len(os.listdir(TASKS_DIR)) if os.path.isdir(TASKS_DIR) else 0}")


## 1. Clean slate

If a previous run left anything behind, this cell clears it out so the run you're about to start begins from a known-good state: it stops the Ray cluster, kills any leftover trainer or SGLang processes, and removes stale session and lock files.

In [ ]:
%%bash
# stop the Ray cluster (head daemons included)
ray stop --force >/dev/null 2>&1 || true
pkill -9 -f 'raylet|sglang|train\.py|run-qwen3-codecontests' 2>/dev/null || true

# clear stale aiter JIT-compile locks
rm -rf /tmp/ray/session_* 2>/dev/null || true
rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true

live() {
  ps -eo stat,args --no-headers \
    | grep -ivE '^Z| grep ' \
    | grep -cE 'raylet|sglang|gcs_server|ray/dashboard|ray\.util|log_monitor|monitor\.py|train\.py|run-qwen3'
}
echo "live ray/sglang/train: $(live)"


## 2. Install miles (editable)

The miles source already ships inside the image, but we install it in editable mode so that any local edits you make to the bind-mounted repo take effect immediately, without a rebuild.

In [ ]:
%%bash
cd "$REPO_DIR" && pip install -e . --no-deps --no-build-isolation -q
python3 -c 'import miles, miles_plugins.mbridge' && echo 'miles import ok'


## 3. Start Harbor (background)

This cell launches the Harbor agent server in the background and waits for it to report healthy at `localhost:11000/health`.

> Harbor's own output doesn't print in this cell — it's redirected to `$WORK_DIR/harbor.log` so the cell can stay focused on reporting a health check and a short log tail. If you want to follow Harbor's log as it runs, open a terminal and run `tail -f $WORK_DIR/harbor.log`.

In [ ]:
%%bash
export MSWEA_CONFIG_FILE="$EX/harbor/codecontests.yaml" \
       HARBOR_TASKS_DIR="$HARBOR_TASKS_DIR" \
       HARBOR_TRIALS_DIR="${HARBOR_TRIALS_DIR:-$WORK_DIR/cc_trials}" \
       HARBOR_DELETE_CONTAINERS=true \
       OPENAI_API_KEY=dummy MSWEA_API_KEY=dummy
# (HARBOR_ENV_TYPE defaults to 'subprocess' in the image.)

LOG="$WORK_DIR/harbor.log"

if curl -sf http://localhost:11000/health >/dev/null 2>&1; then
  echo "Harbor already running on http://localhost:11000"
else
  cd "$EX" && PYTHONPATH="$REPO_DIR" setsid "${HARBOR_PYTHON:-python3}" \
      harbor/server.py --port 11000 --max-concurrent 8 </dev/null >"$LOG" 2>&1 &
  echo "Starting Harbor (pid $!) — logging to: $LOG"

  for i in $(seq 1 30); do
    curl -sf http://localhost:11000/health >/dev/null 2>&1 && { echo "Harbor healthy on http://localhost:11000"; break; }
    [ "$i" = 30 ] && echo "Harbor did NOT become healthy in 60s — see the log tail below"
    sleep 2
  done
fi

echo "----- harbor.log (last 10 lines) -----"
tail -n 10 "$LOG" 2>/dev/null || echo "(no log yet)"


## 4. Data preparation (baked)

Because the CodeContests tasks and their difficulty splits are already baked into the image, this cell normally has nothing to do. You'd only run it to rebuild the dataset from scratch — say, to use a different curriculum — in which case it downloads from Hugging Face and re-extracts the tasks.

In [ ]:
%%bash
if [ -d "$HARBOR_TASKS_DIR/code_contests-0000" ] && [ -f "$EX/data/cc_train_easy.jsonl" ]; then
  echo "data present (tasks=$(ls "$HARBOR_TASKS_DIR" | wc -l)); skipping prep"
else
  python3 "$EX/data_prep/extract_codecontests.py" --dataset open-thoughts/CodeContests --out "$HARBOR_TASKS_DIR"
  python3 "$EX/data_prep/split_by_difficulty.py" --tasks "$HARBOR_TASKS_DIR" --out-dir "$EX/data"
fi


### Task directory structure

Each task's `instruction.md` file *is* the prompt the agent receives. The agent (mini-swe-agent) is told to write a Python solution to `/app/solution.py` that reads from stdin and writes to stdout, test it against the sample input, and then submit by issuing `echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT`. The grader, `tests/test.sh`, then runs `python3 /app/solution.py` against a set of hidden tests using pytest: if every test passes, `reward.txt` is written as `1`, otherwise `0`.

Here's what a task directory looks like — this is what Harbor serves out of `$TASKS_DIR/<task_id>/`:
```
code_contests-12701/
├── instruction.md          # the prompt (problem + I/O spec + "write /app/solution.py")
├── environment/Dockerfile  # task env spec (ignored here — graded on the host)
├── task.toml               # task metadata
└── tests/
    ├── test.sh             # grader: runs solution.py, writes reward.txt (1/0)
    ├── test_state.py       # pytest assertions
    └── test_data.json      # hidden test cases
```

And here's what a trial produces once training (§6) starts running: each attempt lands in `$WORK_DIR/cc_trials/<task_id>__<id>/`:
```
agent/mini-swe-agent.txt              # full turn-by-turn transcript (human-readable)
agent/mini-swe-agent.trajectory.json  # machine-readable messages
verifier/reward.txt                   # 1 (passed) or 0
result.json                           # trial outcome
```

## 5. Model

This cell pre-fetches `Qwen/Qwen3-1.7B` into the HF cache (`$HF_CACHE`, which defaults to `runtime/cache/hf` and is exposed via `HF_HOME`), so the weights are already local by the time training starts.

Running this cell is optional — the training launcher in the next section downloads the same weights itself if they aren't already cached. Running this cell first just separates the ~3.4 GB download from that first training run.

In [ ]:
%%bash
echo "Fetching Qwen/Qwen3-1.7B into the HF cache (HF_HOME=$HF_HOME)..."
python3 -c 'from huggingface_hub import snapshot_download; print(snapshot_download("Qwen/Qwen3-1.7B"))'
echo "weights ready (cache size: $(du -sh "$HF_HOME" 2>/dev/null | cut -f1))"


## 6. Training run

This launches the GRPO loop — rollout, reward, update, weight-sync, checkpoint — all within this container. Each rollout is graded by the Harbor server on `localhost:11000`. The run is launched detached, so once you kick it off, switch to the monitor cell below to watch it progress.

Only run one training job at a time. If an earlier run is still live, reset it first using §8.

> Training's own output doesn't appear in this cell either — everything goes to `$WORK_DIR/train.log`, and the launch cell stays quiet on purpose so you can keep using the notebook while it runs. Use the monitor cell (§7) to watch progress, or run `tail -f $WORK_DIR/train.log` from a terminal.


In [ ]:
%%bash
cd "$EX"

# pin loopback IPs: without this, miles probes the LAN IP and every model call
# from the proot-sandboxed agent fails
export WANDB_KEY="$WANDB_KEY" \
       MILES_HOST_IP="${MILES_HOST_IP:-127.0.0.1}" \
       MILES_ROUTER_EXTERNAL_HOST="${MILES_ROUTER_EXTERNAL_HOST:-127.0.0.1}" \
       AGENT_SERVER_URL=http://localhost:11000 \
       HARBOR_TASKS_DIR="$HARBOR_TASKS_DIR" \
       WANDB_DIR="$WORK_DIR/wandb"

LOG="$WORK_DIR/train.log"

# clear a stale aiter lock, but only if nothing is compiling right now
if ! pgrep -f 'sglang::|run-qwen3-codecontests|train\.py' >/dev/null 2>&1; then
  rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true
fi

PYTHONPATH="$REPO_DIR" setsid python3 run-qwen3-codecontests.py \
    --prompt-data "$EX/data/cc_train_easy.jsonl" \
    --num-rollout 20 \
    --rollout-batch-size 2 \
    --n-samples-per-prompt 8 \
    --global-batch-size 16 \
    --max-seq-len 16384 --save-interval 2 </dev/null >"$LOG" 2>&1 &

echo "Training launched in the background (pid $!)."
echo "  Logs:        $LOG"
echo "  Watch live:  run the monitor cell (below), or in a terminal:  tail -f $LOG"
echo "  Startup takes a few minutes (Megatron init + SGLang engine load) before the first rollout."


## 7. Monitoring training

A single cell drives the whole monitoring view. On every refresh it prints the W&B run link (parsed from `train.log`, pinned at the top so it doesn't scroll away) followed by a live tail of `train.log`. What it shows below that depends on where the run currently is:

- During the rollout phase, before the first training step has finished, it reuses the rollout monitor (`rollout_panel` from `notebook-monitoring/`) to show a per-sample view of turns and rewards.
- Once the first full rollout and training step complete, it switches over to live, hover-able line charts.

Those charts are rendered directly in the notebook, using data parsed from `train.log`, so you see the same curves whether or not you're also looking at W&B:

- **Timing** — `step_time` (`perf/step_time`) plotted against `rollout_time` (`perf/rollout_time`).
- **Reward** — `rollout/raw_reward`, the pass rate, per step.
- **Truncated** — `rollout/truncated` per step.

The charts use plotly (installed automatically on first run) with `hovermode="x unified"`, so hovering any point shows the Y value for every series at that step. Below the charts you'll also find a per-step table, reprinted on every refresh so no step is ever dropped from view, and the turns-per-sample breakdown for the most recent rollout. The cell re-renders in place as it goes; interrupting it (the stop button) only stops the cell from reading logs — it has no effect on the training run itself.

In [ ]:
# Live training monitor: W&B link + train.log tail + current-step rollout panel +
# hover-able charts. Definitions live in notebook-monitoring/training_monitor.py. Interrupt to end.
import os, sys, importlib
sys.path.insert(0, os.path.join(
    os.environ.get("EX", os.path.join(os.getcwd(), "examples/experimental/qwen3-codecontests")),
    "notebook-monitoring"))
import rollout_monitor, training_monitor
importlib.reload(rollout_monitor); importlib.reload(training_monitor)
training_monitor.watch_training()


W&B run: (link not in train.log yet)

=== train.log (raw tail, last 5) ===
(RolloutManager pid=18970) [2026-07-08 20:22:50] sessions.py:58 - [session-server] REQUEST ARRIVED: POST /sessions/6fcdc0bc5a234e78a27d0c433c3da9ba/v1/chat/completions from=127.0.0.1:32864 inflight_chat=4
(RolloutManager pid=18970) [2026-07-08 20:22:50] sessions.py:64 - [session-server] REQUEST DONE: POST /sessions/6fcdc0bc5a234e78a27d0c433c3da9ba/v1/chat/completions status=400 elapsed=0.001s from=127.0.0.1:32864
(SGLangEngine pid=19108) [2026-07-08 20:22:51] Decode batch, #running-req: 4, #token: 45228, token usage: 0.04, cuda graph: True, gen throughput (token/s): 664.52, #queue-req: 0
(SGLangEngine pid=19108) [2026-07-08 20:22:51] Decode batch, #running-req: 4, #token: 45388, token usage: 0.04, cuda graph: True, gen throughput (token/s): 664.95, #queue-req: 0
(SGLangEngine pid=19108) [2026-07-08 20:22:51] Decode batch, #running-req: 4, #token: 45548, token usage: 0.04, cuda graph: True, gen throughput (token/

step  step_t  roll_t  reward  trunc
   0     428     284   0.312  0.562
   1     295     268   0.188  0.062
   2     698     646   0.188  0.500
   3     391     361   0.125  0.500
step_t = perf/step_time reported by the trainer for each step (its own measurement, including step 0); not wall-clock timed by this monitor.


## 8. Stop / reset between runs

Only run one rollout/training job at a time. To reset between runs, this cell kills the Ray/SGLang/trainer processes and clears their session directories — the same cleanup as §1. If you're still seeing endpoint errors after resetting, the safest fix is to relaunch the whole workshop environment. Note that this cell doesn't touch Harbor, since it's a separate process — if you killed Harbor too, re-run §3 to bring it back.

In [ ]:
%%bash
# stop the Ray cluster (head daemons included), then the trainer
ray stop --force >/dev/null 2>&1 || true
pkill -9 -f 'raylet|sglang|train\.py|run-qwen3-codecontests' 2>/dev/null || true
sleep 3

# clear session + JIT lock files
rm -rf /tmp/ray/session_* 2>/dev/null || true
rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true

live() {
  ps -eo stat,args --no-headers \
    | grep -ivE '^Z| grep ' \
    | grep -cE 'raylet|sglang|gcs_server|ray/dashboard|ray\.util|log_monitor|monitor\.py|train\.py|run-qwen3'
}
echo "after reset; live ray/sglang/train: $(live)"
echo 'NOTE: Harbor was NOT killed (different process); re-run the Start Harbor cell only if you killed it.'
